# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = one content item (content_hash_id), for one client (client_hash_id), aggregated over one calendar month. The raw data ships at daily grain (one row per client + content item + day); I roll this up to monthly by summing volume metrics (impressions, clicks, sessions, etc.) and recalculating gsc_avg_position as a weighted average (sum(gsc_sum_position) / sum(gsc_impressions)) rather than averaging daily averages, since the latter would misweight low-traffic days equally with high-traffic days.

**Time window:** Working month = 2026-03, chosen as a mid-panel month for iteration per the assignment's instructions. The dataset's fact_content_daily_performance_sample file (June 2026, the final available month) is treated as a sealed test month and is never used to develop label or feature logic — only to sanity-check that queries run.

**Verification:** Confirmed the raw daily table has zero duplicate rows per (client, content, date) — 9,841,378 unique combos = 9,841,378 total rows. After monthly aggregation, row count (331,437) exactly matches the count of unique client+content pairs in the raw March data, confirming no rows were lost or duplicated in the rollup. 154,699 rows have zero GSC impressions in March, which correctly produces NaN for gsc_avg_position (undefined average position with no impressions) rather than a computation error.

In [34]:
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

In [35]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"

# Flat files — load directly
df_content = con.sql(f"SELECT * FROM read_parquet('{base}/dim_content.parquet')").df()
df_clients = con.sql(f"SELECT * FROM read_parquet('{base}/dim_clients.parquet')").df()

# March 2026 — go straight to that month's file, no filtering needed
df_march = con.sql(f"SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')").df()

print(f"dim_content: {df_content.shape}")
print(f"dim_clients: {df_clients.shape}")
print(f"March daily rows: {df_march.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Check: does one row in df_march = one client + one content item + one day?
duplicate_check = df_march.groupby(['client_hash_id', 'content_hash_id', 'report_date']).size()
print(f"Any duplicate rows for the same client+content+date? {(duplicate_check > 1).sum()} duplicates found")
print(f"Total unique client+content+date combos: {len(duplicate_check)}")
print(f"Total rows in df_march: {len(df_march)}")

In [ ]:
df_march.columns.tolist()

In [ ]:
agg_df = df_march.groupby(['client_hash_id', 'content_hash_id']).agg(
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    gsc_sum_position=('gsc_sum_position', 'sum'),
    ga4_pageviews=('ga4_pageviews', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_users=('ga4_users', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
    ga4_total_engagement_sec=('ga4_total_engagement_sec', 'sum'),
    sessions_organic=('sessions_organic', 'sum'),
    sessions_direct=('sessions_direct', 'sum'),
    sessions_referral=('sessions_referral', 'sum'),
    sessions_social=('sessions_social', 'sum'),
    sessions_paid=('sessions_paid', 'sum'),
    sessions_ai=('sessions_ai', 'sum'),
    ai_chatgpt=('ai_chatgpt', 'sum'),
    ai_perplexity=('ai_perplexity', 'sum'),
    ai_gemini=('ai_gemini', 'sum'),
    ai_copilot=('ai_copilot', 'sum'),
    ai_claude=('ai_claude', 'sum'),
    ai_meta=('ai_meta', 'sum'),
    ai_other=('ai_other', 'sum'),
    scroll_events=('scroll_events', 'sum'),
    client_has_gsc=('client_has_gsc', 'max'),
    client_has_ga4=('client_has_ga4', 'max'),
    gsc_data_available=('gsc_data_available', 'max'),
    ga4_data_available=('ga4_data_available', 'max'),
    days_with_data=('report_date', 'nunique'),
).reset_index()

# Recalculate avg position correctly (weighted, not average-of-averages)
agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']

agg_df['month'] = '2026-03'

print(f"Monthly rows: {agg_df.shape}")
agg_df.head()

In [ ]:
#checking whether the 0's and NaN's seen mean that they occured due to the division of 0/0
zero_impression_rows = (agg_df['gsc_impressions'] == 0).sum()
nan_position_rows = agg_df['gsc_avg_position'].isna().sum()

print(f"Rows with 0 GSC impressions: {zero_impression_rows}")
print(f"Rows with NaN avg position: {nan_position_rows}")


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context** (needed for joins/filtering, not modeling inputs): client_hash_id, content_hash_id, month, days_with_data, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available

**Feature candidates** (14 — will narrow to 5 in Section 3): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, scroll_events. All are observed signals known by month-end — no future information, no product-computed scores.

**Label or proxy:** Intentionally empty this week. A decline/opportunity label requires comparing at least two months (e.g. prior 90 days → next 30 days per the lane guide), but this contract currently works within a single month (March 2026). Building a real proxy label is out of scope until a second time window is added — documented as a limitation in Section 4.

**Excluded:**

gsc_sum_position — raw intermediate value used only to calculate

gsc_avg_position; not meaningful as a standalone feature on its own.

sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — AI-referral signals are excluded this week. Per the lane guide, only 30,177 of 78.8M daily rows have any AI session data — too sparse to use safely as features without producing misleading results.

In [ ]:
field_buckets = {
    'context': [
        'client_hash_id', 'content_hash_id', 'month', 'days_with_data',
        'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available'
    ],
    'feature': [
        'gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
        'ga4_pageviews', 'ga4_sessions', 'ga4_users',
        'ga4_engaged_sessions', 'ga4_total_engagement_sec',
        'sessions_organic', 'sessions_direct', 'sessions_referral',
        'sessions_social', 'sessions_paid', 'scroll_events'
    ],
    'excluded': [
        'gsc_sum_position',  # raw intermediate used only to compute gsc_avg_position — not a standalone feature
        'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini',
        'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other'
        # AI-referral columns — lane guide flags these as too sparse
        # (30,177 rows with AI sessions out of 78.8M total) to use safely this week
    ],
    'label_or_proxy': []  # intentionally empty — no valid label possible from a single month; addressed in Section 4
}

# sanity check: did we account for every column?
all_bucketed = sum(field_buckets.values(), [])
missing = set(agg_df.columns) - set(all_bucketed)
print(f"Columns not yet bucketed: {missing}")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Five-feature frame:**

gsc_impressions — knowable at decision moment because it's a completed count of March search appearances; measures raw visibility/demand.
gsc_clicks — knowable because it's a completed count of March clicks; measures actual traffic capture.
gsc_avg_position — knowable because it's recalculated purely from March's own position/impression data; measures ranking strength.
ga4_sessions — knowable because it's a completed count of March visits; measures overall traffic volume beyond search alone.
ga4_engaged_sessions — knowable because it's a completed count of visits with real engagement behavior in March; measures content quality/relevance signal, not just traffic.

Chosen to cover four distinct angles — demand, capture, ranking, and quality — without redundant overlap (e.g., excluding ga4_pageviews since it's highly correlated with ga4_sessions).

In [ ]:
#Grain check — one row per client + content pair, for March
grain_check = agg_df.groupby(['client_hash_id', 'content_hash_id']).size()
duplicates = (grain_check > 1).sum()

print(f"Grain claim: one row = one client + one content item + one month")
print(f"Duplicate (client, content) pairs found: {duplicates}")
print(f"Total rows in agg_df: {len(agg_df)}")
print(f"Total unique (client, content) pairs: {len(grain_check)}")
print(f"Grain confirmed: {duplicates == 0 and len(agg_df) == len(grain_check)}")

In [ ]:
#Slice row count and date span
print(f"Raw March rows (daily grain): {len(df_march)}")
print(f"Aggregated March rows (monthly grain): {len(agg_df)}")
print(f"Date range covered: {df_march['report_date'].min()} to {df_march['report_date'].max()}")
print(f"Unique clients in slice: {df_march['client_hash_id'].nunique()}")
print(f"Unique content items in slice: {df_march['content_hash_id'].nunique()}")

In [ ]:
#Availability check with IS TRUE
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS both_available_rows
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

print(availability_check)

In [ ]:
#final 5 features
final_features = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']
feature_frame = agg_df[['client_hash_id', 'content_hash_id'] + final_features].copy()
feature_frame.head()

**The Trap** (label leakage demonstration):

To isolate the leakage effect cleanly, I used a purely random demo label (coin-flip, no real relationship to any feature) rather than a real project label — since a real label doesn't exist yet for this single-month contract (see Section 4).

With the 5 honest features, logistic regression scored an AUC of 0.5013 — essentially random guessing, exactly as expected, since a random label truly cannot be predicted from real signals.

I then added leaky_column, which was the label itself smuggled in disguised as a feature. The AUC jumped to 1.0000 — a +0.4987 increase. This is textbook leakage: the model wasn't learning a pattern, it was looking up the answer directly.

I removed the leaky column and confirmed the honest score (0.5013) as the number to report. Lesson: any feature mathematically identical to (or derived from) the label will produce a fake, inflated score — this is the leakage risk the lane guide warns about when reusing product-computed fields or accidentally encoding the target.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
#TRAP DEMONSTRATION: clean isolation using a random label
import numpy as np

np.random.seed(42)
demo_label = np.random.randint(0, 2, size=len(agg_df))  # pure coin-flip, no real relationship to anything

X_honest = feature_frame[final_features].fillna(0)
y = demo_label

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)

model_honest = LogisticRegression(max_iter=1000)
model_honest.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])

print(f"Honest AUC score (5 real features, random label): {honest_score:.4f}")

In [ ]:
# TRAP: this column is built directly from the label (literal leakage)
X_leaky = X_honest.copy()
X_leaky['leaky_column'] = demo_label  # the label itself, disguised as a "feature"

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(X_leaky, y, test_size=0.2, random_state=42)

model_leaky = LogisticRegression(max_iter=1000)
model_leaky.fit(X_train_leak, y_train_leak)
leaky_score = roc_auc_score(y_test_leak, model_leaky.predict_proba(X_test_leak)[:, 1])

print(f"Leaky AUC score (label smuggled in as a feature): {leaky_score:.4f}")
print(f"Score jump: {leaky_score - honest_score:.4f}")

In [ ]:
# Delete the leaky column, keep the honest number
X_leaky = X_leaky.drop(columns=['leaky_column'])
print("Leaky column removed. Final honest feature set restored.")
print(f"Final honest AUC to report: {honest_score:.4f}")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data limits:**

1. Single-month scope, no real label yet. This contract works within March 2026 only. A genuine decline/opportunity label requires comparing at least two time windows (e.g., prior 90 days → next 30 days, per the lane guide), which is out of scope until a second month is added in later weeks.
2. Unbalanced panel — nearly half of clients are absent from this slice. 49 of 104 total clients (47%) show zero activity in March. This contract's row counts and feature distributions describe only the 55 currently-active clients, not the full client base — any conclusions should be scoped accordingly.
3. GA4 tracking rolled out much later than GSC, and is still sparse. GSC history begins 2025-01-27, while GA4 begins 2025-10-29 — nearly 9 months later platform-wide. Only 51 of 104 clients have a recorded GA4 start date at all, and the 75th-percentile GA4 start date is 2026-03-21 — meaning most GA4-tracked clients only just started being tracked around the observation month itself. This explains why only 4.2% of March rows have usable GA4 data, versus 36.7% for GSC. Any GA4-based feature (ga4_sessions, ga4_engaged_sessions) will have far more missing/zero values than GSC-based features, and low GA4 coverage should not be misread as "no engagement" — it may just mean tracking hadn't started yet.
4. 37 clients have no recorded GSC start date, and 53 have no GA4 start date. These clients' dim_clients history fields are simply empty/null — meaning tracking-start metadata isn't available for a meaningful share of the client base, not just data-in-window itself.
5. Decision-support only, not causal. As shown in the leakage trap (Section 3), a high score proves a model found a pattern — never that the pattern is real-world causal. This data can rank pages by evidence-backed opportunity; it cannot prove a refresh would cause recovery, per the lane guide's explicit warning against causal claims without a valid experiment.

In [ ]:
# How many of the 104 total clients actually appear in March data?
total_clients = df_clients.shape[0]
active_march_clients = df_march['client_hash_id'].nunique()

print(f"Total clients in dim_clients: {total_clients}")
print(f"Clients active in March: {active_march_clients}")
print(f"Clients missing from March slice: {total_clients - active_march_clients}")

In [ ]:
# Check the unbalanced panel claim — do clients have different history start dates?
df_clients[['client_hash_id', 'gsc_data_start', 'ga4_data_start']].describe()

In [ ]:
# GA4 data is much sparser than GSC — confirm the gap already seen in Query 3
ga4_pct = (413966 / 9841378) * 100
gsc_pct = (3611061 / 9841378) * 100
print(f"GSC data available in: {gsc_pct:.1f}% of March rows")
print(f"GA4 data available in: {ga4_pct:.1f}% of March rows")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.